# Collect Data

# Prepare data
## Data preparation - step 1



In [26]:
from pyspark.sql import SparkSession, functions as F, types as T
from pyspark.sql.functions import col

In the following, knowing that there is no header in the files and ";" is the seperator, we specified those options to import and merge the prices files from 2022 to 2024.

In [17]:
spark = (SparkSession.builder
         .appName("Prix")
         .getOrCreate())

files_path = "../Data/1-raw/Prix*.csv.gz"

df_prices = (spark.read
    #.option("header", True)
    .option("inferSchema", True) 
    .option("sep", ";")              
    .csv(files_path)                 
)

df_prices.head(5)

[Row(_c0=1000001, _c1=1000, _c2='R', _c3=4620100.0, _c4=519800.0, _c5=datetime.datetime(2023, 1, 2, 7, 53, 26), _c6=1, _c7='Gazole', _c8=1.867),
 Row(_c0=1000001, _c1=1000, _c2='R', _c3=4620100.0, _c4=519800.0, _c5=datetime.datetime(2023, 1, 5, 9, 33, 37), _c6=1, _c7='Gazole', _c8=1.877),
 Row(_c0=1000001, _c1=1000, _c2='R', _c3=4620100.0, _c4=519800.0, _c5=datetime.datetime(2023, 1, 9, 14, 51, 49), _c6=1, _c7='Gazole', _c8=1.875),
 Row(_c0=1000001, _c1=1000, _c2='R', _c3=4620100.0, _c4=519800.0, _c5=datetime.datetime(2023, 1, 11, 9, 23, 54), _c6=1, _c7='Gazole', _c8=1.859),
 Row(_c0=1000001, _c1=1000, _c2='R', _c3=4620100.0, _c4=519800.0, _c5=datetime.datetime(2023, 1, 13, 9, 7, 40), _c6=1, _c7='Gazole', _c8=1.862)]

In the read.me, we saw the name of the columns and their descriptions. We just included them in order to rename our columns.

In [ ]:
cols = [
    "id_pdv", "cp", "pop",
    "latitude", "longitude",
    "date",
    "id_carburant", "nom_carburant",
    "prix"
]

df_prices = df_prices.toDF(*cols)
df_prices.show(5)
df_prices.printSchema()

+-------+----+---+---------+---------+-------------------+------------+-------------+-----+
|id_pdv |cp  |pop|latitude |longitude|date               |id_carburant|nom_carburant|prix |
+-------+----+---+---------+---------+-------------------+------------+-------------+-----+
|1000001|1000|R  |4620100.0|519800.0 |2023-01-02 07:53:26|1           |Gazole       |1.867|
|1000001|1000|R  |4620100.0|519800.0 |2023-01-05 09:33:37|1           |Gazole       |1.877|
|1000001|1000|R  |4620100.0|519800.0 |2023-01-09 14:51:49|1           |Gazole       |1.875|
|1000001|1000|R  |4620100.0|519800.0 |2023-01-11 09:23:54|1           |Gazole       |1.859|
|1000001|1000|R  |4620100.0|519800.0 |2023-01-13 09:07:40|1           |Gazole       |1.862|
+-------+----+---+---------+---------+-------------------+------------+-------------+-----+
only showing top 5 rows

root
 |-- id_pdv: integer (nullable = true)
 |-- cp: integer (nullable = true)
 |-- pop: string (nullable = true)
 |-- latitude: double (nullable =

Knowing the previous format of the files, we use the withColumn methods to extract the year, the month and the week of the year.

In [ ]:
df_prices = (df_prices
    .withColumn("year",  F.year("date"))
    .withColumn("month", F.month("date"))
    .withColumn("week_of_year", F.weekofyear("date"))
)

df_prices.select("date","year","month","week_of_year").show(5)

+-------------------+----+-----+------------+
|date               |year|month|week_of_year|
+-------------------+----+-----+------------+
|2023-01-02 07:53:26|2023|1    |1           |
|2023-01-05 09:33:37|2023|1    |1           |
|2023-01-09 14:51:49|2023|1    |2           |
|2023-01-11 09:23:54|2023|1    |2           |
|2023-01-13 09:07:40|2023|1    |2           |
+-------------------+----+-----+------------+
only showing top 5 rows



In [ ]:
df_prices.filter(col("prix").isNull).show()

In [ ]:
# We divide the columns longitude and latitude by 10^5
# as it was making sense for the first one and keeping five zeros is common in order to get a 1-meter precision 
df_prices = (df_prices 
.withColumn("latitude", (F.col("latitude") / 100000)) 
.withColumn("longitude", (F.col("longitude") / 100000))
)

df_prices.select("latitude", "longitude").show(5)




+--------+---------+
|latitude|longitude|
+--------+---------+
|46.201  |5.198    |
|46.201  |5.198    |
|46.201  |5.198    |
|46.201  |5.198    |
|46.201  |5.198    |
+--------+---------+
only showing top 5 rows



In [ ]:
# At this step we wanted to check if we got some NA values or some null values
print(df_prices.count()) # 14 214 837
df_prices.filter(col("prix").isNull()).count()/df_prices.count() # less than 1% is na so we will drop them
df_prices = df_prices.na.drop()

14214837


0.000897372231563401

In [24]:
# In order to make the table available to spark SQL we use the the createOrReplaceTempView method
# We rename it prices_SQL
df_prices.createOrReplaceTempView('prices_SQL')

# We now will try to the distribution of each gas type in order to choose the two out of our interest
spark.sql("""
    SELECT nom_carburant,
        COUNT(nom_carburant) AS freq
    FROM prices_SQL
    GROUP BY nom_carburant
""").show()

+-------------+-------+
|nom_carburant|   freq|
+-------------+-------+
|          E10|3559498|
|         SP98|3425844|
|         NULL|      0|
|          E85|1390580|
|       Gazole|4245380|
|         SP95| 961020|
|         GPLc| 619759|
+-------------+-------+



# Visualize gas prices

# Model gas prices evolution

# Evaluate if electric cars development as an impact